In [ ]:
# ============================================================
# Cell 1 — Install Dependencies
# ============================================================
!pip install -q \
    unsloth \
    transformers \
    accelerate \
    bitsandbytes \
    huggingface_hub \
    datasets \
    pandas \
    tqdm \
    pillow \
    scikit-learn \
    nltk \
    rouge-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 861.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 600.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.2/924.2 kB 31.3 MB/s eta 0:00:00

In [ ]:
#============================================================
# Cell 2 — Imports
# ============================================================
import os
import gc
import json
import torch
import numpy as np
import pandas as pd
import nltk
from tqdm import tqdm
from PIL import Image
from huggingface_hub import login
from google.colab import userdata, files
from datasets import load_dataset
from unsloth import FastVisionModel
from transformers import TextStreamer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    jaccard_score,
)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


True

In [ ]:
# Cell 3: Authentication & GPU Cleanup
import torch
import gc
from huggingface_hub import login
from google.colab import userdata

# Clean VRAM
gc.collect()
torch.cuda.empty_cache()

# Login via Colab Secrets
try:
    hf_token = userdata.get('hf_key')
    login(token=hf_token)
    print("✅ Logged in to Hugging Face successfully via Colab Secrets!")
except Exception as e:
    print("❌ Error: 'hf-rw' not found in Colab Secrets. Please add it via the Key icon on the left.")

✅ Logged in to Hugging Face successfully via Colab Secrets!


In [ ]:
import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

In [ ]:
# Cell 4: Load Fine-Tuned Llama Model via Unsloth
from unsloth import FastVisionModel

# ⚠️ REPLACE with your actual fine-tuned Llama repository name
adapter_model_id = "bappy2001/Llama-3.2-11B-ECG-LM-Final_V3"

print("Loading Llama-3.2-11B Vision Model & Tokenizer...")
model, tokenizer = FastVisionModel.from_pretrained(
    adapter_model_id,
    load_in_4bit=True,                 # 🎯 OOM ক্র্যাশ এড়াতে True করা হয়েছে
    torch_dtype=torch.float16,         # 🎯 T4 GPU-এর সাথে সামঞ্জস্য রাখতে fp16 করা হয়েছে
    use_gradient_checkpointing="unsloth",
    token=hf_token,
)

# Switch model to inference mode (Crucial for VRAM optimization on T4)
FastVisionModel.for_inference(model)
print("✅ Model loaded and set to inference mode.")

Loading Llama-3.2-11B Vision Model & Tokenizer...
==((====))==  Unsloth 2026.5.10: Fast Mllama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/375k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/55.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

✅ Model loaded and set to inference mode.


In [ ]:
# Cell 5: Single Image Inference Function
from PIL import Image

def generate_ecg_report(image_path, patient_profile_text):
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return f"Error loading image: {e}"

    # Blank leads to force visual diagnosis
    blank_leads = (
        "lead I: , lead II: , lead III: , lead aVR: , lead aVL: , lead aVF: , "
        "lead V1: , lead V2: , lead V3: , lead V4: , lead V5: , lead V6: "
    )

    user_prompt = (
        f"{patient_profile_text}. "
        f"Analyze the ECG image and generate a complete clinical report. "
        f"Some leads are masked. Use visual evidence to complete them.\n\n"
        f"Visible Leads: {blank_leads}"
    )

    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": user_prompt}
        ]}
    ]

    # Format using Unsloth's integrated tokenizer
    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(images=image, text=text_prompt, return_tensors="pt").to("cuda")

    print("Analyzing ECG Image... 🧠")
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=768,
            use_cache=True,
            temperature=1.5,
            min_p=0.1
        )

    # Slice out the prompt tokens to get only the response
    input_len = inputs["input_ids"].shape[-1]
    generated_tokens = generated_ids[0][input_len:]
    decoded_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    # Free up memory
    del inputs, generated_ids
    torch.cuda.empty_cache()

    return decoded_output

# ==========================================
# Example Run
# ==========================================
print("\n--- Starting Model Test ---")
example_patient_profile = "Patient profile: 55Y male, 75.0kg"
example_image_path = "/content/sample_data/ecg_1.png"  # ⚠️ Ensure this file exists in Colab

# Uncomment the line below to run the test if the image is uploaded
# result = generate_ecg_report(example_image_path, example_patient_profile)
# print("\n==== 🎯 MODEL PREDICTION ====\n", result.strip())


--- Starting Model Test ---


In [ ]:
# Cell 5: Batch Inference (200 Samples) & Saving
import pandas as pd
import json
from tqdm import tqdm
from datasets import load_dataset
from google.colab import files

print("Loading Test Dataset...")
dataset = load_dataset("kazi420/ECG_LM_Final_Dataset_V3", split="test")

def get_patient_profile_and_truth(text_sequence):
    text_sequence = text_sequence.replace("<ECG_IMAGE_TOKEN>", "").strip()
    if "Findings:" in text_sequence:
        patient_profile, ground_truth = text_sequence.split("Findings:", 1)
        return patient_profile.strip(), "Findings: " + ground_truth.strip()
    else:
        return text_sequence.split(".")[0], text_sequence

print("\n🚀 Starting Inference on 200 Images...")
results = []
TOTAL_TEST_SAMPLES = 200

for i, item in enumerate(tqdm(dataset.select(range(TOTAL_TEST_SAMPLES)))):
    original_text = item["text_sequence"]

    if item.get("ecg_image") is not None:
        image = item["ecg_image"].convert("RGB")
    else:
        continue

    patient_profile_text, ground_truth_report = get_patient_profile_and_truth(original_text)

    blank_leads = (
        "lead I: , lead II: , lead III: , lead aVR: , lead aVL: , lead aVF: , "
        "lead V1: , lead V2: , lead V3: , lead V4: , lead V5: , lead V6: "
    )

    user_prompt = (
        f"{patient_profile_text}. Analyze the ECG image and generate a complete clinical report. "
        f"Some leads are masked. Use visual evidence to complete them.\n\n"
        f"Visible Leads: {blank_leads}"
    )

    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_prompt}]}]
    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(images=image, text=text_prompt, return_tensors="pt").to("cuda")

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=768,
            use_cache=True,
            temperature=1.5,
            min_p=0.1
        )

    generated_tokens = generated_ids[0][input_len:]
    prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    results.append({
        "ID": i + 1,
        "Patient_Profile": patient_profile_text,
        "Ground_Truth_Original": ground_truth_report,
        "Model_Prediction": prediction
    })

    del inputs, generated_ids, image
    torch.cuda.empty_cache()

# Save and Download
df = pd.DataFrame(results)
csv_filename = "llama_ecg_200_test_results.csv"
json_filename = "llama_ecg_200_test_results.json"

df.to_csv(csv_filename, index=False)
with open(json_filename, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("\n✅ Testing Complete! Downloading files...")
try:
    files.download(csv_filename)
    files.download(json_filename)
except Exception as e:
    print(f"⚠️ Auto-download failed. You can manually download from Colab file explorer. Error: {e}")

Loading Test Dataset...


README.md:   0%|          | 0.00/588 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/355M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/44.0M [00:00<?, ?B/s]

...


✅ Testing Complete! Downloading files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 6: Metrics Evaluation
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, accuracy_score, jaccard_score
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Load the generated predictions
file_path = '/content/llama_ecg_200_test_results.csv'
df = pd.read_csv(file_path)

def extract_superclasses(text):
    if not isinstance(text, str):
        return ['UNKNOWN']
    text = text.lower()
    classes = []
    if any(w in text for w in ['normal ecg', 'norm']): classes.append('NORM')
    if any(w in text for w in ['infarction', 'mi']): classes.append('MI')
    if any(w in text for w in ['st ', 't wave', 'ischemi', 'isc', 'st-']): classes.append('STTC')
    if any(w in text for w in ['block', 'ivcd', 'delay', 'fascicular']): classes.append('CD')
    if any(w in text for w in ['hypertrophy', 'lvh', 'enlargement', 'lao']): classes.append('HYP')
    return classes if classes else ['UNKNOWN']

y_true, y_pred, ground_texts, pred_texts = [], [], [], []

for index, row in df.iterrows():
    gt = str(row.get('Ground_Truth_Original', ''))
    pred = str(row.get('Model_Prediction', ''))
    y_true.append(extract_superclasses(gt))
    y_pred.append(extract_superclasses(pred))
    ground_texts.append(gt)
    pred_texts.append(pred)

# Classification Metrics
mlb = MultiLabelBinarizer(classes=['NORM', 'MI', 'STTC', 'CD', 'HYP'])
y_true_bin = mlb.fit_transform(y_true)
y_pred_bin = mlb.fit_transform(y_pred)

print("==== 🏆 CLASSIFICATION METRICS (Disease Detection) ====\n")
print(classification_report(y_true_bin, y_pred_bin, target_names=mlb.classes_, zero_division=0))
print(f"► Strict Exact Match Accuracy: {accuracy_score(y_true_bin, y_pred_bin):.4f}")
print(f"► Normalized Partial Accuracy: {jaccard_score(y_true_bin, y_pred_bin, average='samples', zero_division=0):.4f}\n")

# LLM Text Generation Metrics
print("==== 🧠 LLM TEXT EVALUATION METRICS ====\n")

# Cosine
vectorizer = TfidfVectorizer()
cosine_scores = [cosine_similarity(vectorizer.fit_transform([gt, pred])[0:1], vectorizer.fit_transform([gt, pred])[1:2])[0][0] for gt, pred in zip(ground_texts, pred_texts) if gt.strip() and pred.strip()]
print(f"► Average Cosine Similarity: {np.mean(cosine_scores):.4f}")

# Jaccard Text
def text_jaccard_similarity(str1, str2):
    set1, set2 = set(str1.lower().split()), set(str2.lower().split())
    union = set1.union(set2)
    return len(set1.intersection(set2)) / len(union) if union else 0
print(f"► Average Text Jaccard Similarity: {np.mean([text_jaccard_similarity(gt, pred) for gt, pred in zip(ground_texts, pred_texts)]):.4f}")

# BLEU
smoother = SmoothingFunction().method1
bleu_scores = [sentence_bleu([nltk.word_tokenize(gt.lower())], nltk.word_tokenize(pred.lower()), smoothing_function=smoother) for gt, pred in zip(ground_texts, pred_texts)]
print(f"► Average BLEU Score: {np.mean(bleu_scores):.4f}")

# ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
rouge1, rougeL = zip(*[(scorer.score(gt, pred)['rouge1'].fmeasure, scorer.score(gt, pred)['rougeL'].fmeasure) for gt, pred in zip(ground_texts, pred_texts)])
print(f"► Average ROUGE-1 F1 Score: {np.mean(rouge1):.4f}")
print(f"► Average ROUGE-L F1 Score: {np.mean(rougeL):.4f}")
print("====================================================")

==== 🏆 CLASSIFICATION METRICS (Disease Detection) ====

              precision    recall  f1-score   support

        NORM       1.00      0.95      0.98       200
          MI       0.53      0.31      0.39        99
        STTC       0.60      0.67      0.63       109
          CD       0.39      0.43      0.41        88
         HYP       0.36      0.30      0.33        67

   micro avg       0.67      0.63      0.65       563
   macro avg       0.58      0.53      0.55       563
weighted avg       0.67      0.63      0.64       563
 samples avg       0.73      0.69      0.64       563

► Strict Exact Match Accuracy: 0.0800
► Normalized Partial Accuracy: 0.5007

==== 🧠 LLM TEXT EVALUATION METRICS ====

► Average Cosine Similarity: 0.7688
► Average Text Jaccard Similarity: 0.3332
► Average BLEU Score: 0.4574
► Average ROUGE-1 F1 Score: 0.6040
► Average ROUGE-L F1 Score: 0.5479


In [ ]:
# Cell 7: Inference Benchmarking & Cost Estimation
import time
import torch
from PIL import Image

# 1. CRITICAL: Switch model to inference mode to free up VRAM
FastVisionModel.for_inference(model)

# 2. Define your inputs
# Make sure to upload an image to Colab and point to it here
test_image_path = "/content/sample_data/ecg_1.png"  # ⚠️ Update with your actual filename
patient_context = "Patient Profile: 45 yo male, 80kg"

print("Starting inference benchmark on T4...")

# 3. Measure inference time for one sample using the function defined in Cell 4
start = time.time()
report = generate_ecg_report(test_image_path, patient_context)
end = time.time()

# 4. Calculate metrics
inference_time_seconds = end - start
cost_per_hour_t4 = 0.526  # AWS g4dn.xlarge USD
cost_per_report = (inference_time_seconds / 3600) * cost_per_hour_t4

print("\n--- Benchmark Results ---")
print(f"Inference time: {inference_time_seconds:.2f} seconds")
print(f"Cost per report (T4): ${cost_per_report:.5f}")
print(f"Reports per hour: {3600/inference_time_seconds:.0f}")

# Optional: Print a snippet of the generated report to verify it worked
print("\nGenerated Report Snippet:")
print(report[:200] + "...")

Starting inference benchmark on T4...

--- Benchmark Results ---
Inference time: 0.00 seconds
Cost per report (T4): $0.00000
Reports per hour: 1428659

Generated Report Snippet:
Error loading image: [Errno 2] No such file or directory: '/content/sample_data/ecg_1.png'...
